## 1 · Data download and problem definition

### By:
Sofía Arango Patiño

### Date:
2026-10-31

### Description:

First step of the proof of concept ([issue #1](https://github.com/modsiw-mona/anomaly_detection_ato/issues/1);
CRISP-DM phase 1, *business understanding*, and the start of phase 2; thesis objective OE1).

This notebook:

1. frames the problem by answering the 14 problem-definition questions of the course methodology;
2. downloads the **Login Data Set for Risk-Based Authentication** reproducibly — fixed source, verified
   checksum, archive inspected before extraction — into `data/01_raw/`;
3. takes a first, *structural* look at the raw file: columns, size, and whether it meets the project's
   minimum data requirements.

Detailed exploration (types, nulls, distributions) is issue #2; the analysis of the data is issue #3.

All decisions referenced here are recorded in [`docs/development-plan.md`](../../docs/development-plan.md);
the dataset itself is described in [`docs/datasets/rba.md`](../../docs/datasets/rba.md).

## 🧭 Problem definition

**1. What is the objective of the problem?**

To detect **account takeover (ATO)** — an attacker logging in to someone else's account — by automatically
identifying *behavioural anomalies* in authentication records. The output is a **risk score per login
attempt**, so that suspicious logins are acted on before the attacker can do damage, reducing the time an
account is exposed (anteproyecto, general objective 6.1).

**2. How will the solution be used?**

As in risk-based authentication (RBA): every login is scored, and the response is proportional to the risk.
A low score lets the user in; a medium score asks for extra verification (step-up, e.g. a one-time code); a
high score blocks the attempt or sends it to the security team. Within the thesis it is evaluated offline and
demonstrated with a prototype that scores a single login (OE4, issue #8).

**3. What are the current solutions (if any)?**

- **Static rules** (e.g. "block after N failures", "alert on a new country"). They are what most systems still
  run, and they fall behind as attack techniques change (Ren, 2026).
- **Risk-based authentication**, notably the statistical model of Freeman et al. (2016), which compares each
  login's features with the user's own history and with the global population. It is deployed by large
  online services and is the reference model for this dataset (Wiefling et al., 2022).
- **Commercial and UEBA tools** (user and entity behaviour analytics), e.g. Amazon Fraud Detector's
  account-takeover model, which score login events from behavioural features.

**4. How should the problem be framed (supervised / unsupervised, online / offline, ...)?**

- **Unit of analysis:** one login attempt (a row), judged against the history of the same user.
- **Labels exist but are scarce and incomplete.** `Is Account Takeover` marks the takeovers the service's
  incident response team *confirmed*: 140 successful logins in the published file (the companion paper
  reports 87 for the original data; see the counts below). A login
  not flagged is *unlabelled*, not necessarily legitimate. This is a **positive-unlabelled (PU)** setting with
  extreme imbalance.
- Therefore the problem is framed as **anomaly detection with weak supervision**: models that score how
  unusual a login is for its user, with the few confirmed labels used to evaluate, compare and calibrate —
  and supervised classifiers compared on equal terms (OE3), using repeated cross-validation instead of a
  single split.
- **Offline training, per-event scoring.** Models are trained in batch on historical data; scoring a single
  login is cheap and can happen at login time.

**5. How should performance be measured — a first intuition?**

**Accuracy is useless here:** a model that says "legitimate" to every login is right more than 99.99% of the
time and catches nothing. Instead:

- **PR-AUC** (area under the precision-recall curve) — the primary metric, threshold-independent;
- **Recall at an alert budget** — the share of takeovers caught if the security team can only review, say,
  the top 0.1% or 1% of logins;
- **Precision**, reported as a *lower bound*, because some "false positives" may be takeovers nobody caught;
- **Friction** — the share of legitimate logins that would be challenged at the chosen threshold.

**6. Is the performance measure aligned with the objective?**

Yes. Recall measures takeovers caught, which is what shortens exposure. Precision and friction measure the
cost paid by legitimate users and analysts, which decides whether a model is usable at all. The alert budget
ties the evaluation to a real operational capacity, and PR-AUC summarises the trade-off without committing to
a threshold too early.

**7. What is the minimum performance needed to reach the objective?**

There is no published threshold for this dataset. The minimum bar is to **beat the established method** —
the Freeman et al. (2016) RBA score — and a dummy model, on PR-AUC and on recall at the same alert budget,
with confidence intervals that support the difference. A concrete operating target (recall at, e.g., at most
1% of legitimate logins challenged) will be fixed once the baseline exists (issue #5), so the target is
grounded in data rather than guessed.

**8. What are similar problems? Can existing experience or tools be reused?**

Similar problems: transactional fraud detection (credit cards), network intrusion detection, insider-threat
detection, lateral-movement detection in enterprise authentication logs. Reusable assets:

- the Freeman RBA algorithm and its reference implementation (`das-group/rba-algorithm`, read as text and
  re-implemented here as the baseline);
- anomaly-detection libraries (PyOD) and gradient-boosting libraries for tabular data;
- current practice for extreme imbalance: class weights and threshold tuning, PU learning, conformal
  calibration of false-alarm rates.

**9. Is experience with the problem available?**

Yes, from two sides:

- **Practical (author):** the author is a cybersecurity engineer focused on application security (AppSec),
  with experience building security tools. That experience is not in fraud or network detection
  specifically, but it covers the attack surface account takeover exploits: authentication and session
  management weaknesses (OWASP Top 10, *A07 — Identification and Authentication Failures*) and the attacks
  against them, such as credential stuffing and session hijacking. It informs the "how would an analyst solve
  this manually" reasoning in question 10 and the choice of behavioural signals.
- **Published:** the dataset was released together with a peer-reviewed study of the Freeman RBA model on the
  original data (Wiefling et al., 2022), and later work has trained detection models on the same public
  dataset (e.g. Sasikumar & Nagarajan, 2026). The anteproyecto's literature review summarises this
  experience.

**10. (Important) How could the problem be solved manually?**

A security analyst looking at one login would ask:

- Is this **device / browser** new for this account?
- Is the **network (ASN)** or **country** new for this user — and is it rare across all users?
- Were there **failed attempts on this account just before** this success (credential stuffing)?
- Does the **IP** appear in known-attacker lists, or is it **shared by many accounts**?
- Is the **time** unusual compared with the user's habits?

Several of these at once make a login suspicious. This manual procedure is exactly the logic of risk-based
authentication, and each question becomes a *behavioural feature* in issue #4.

**11. List of assumptions so far**

1. `Is Account Takeover` is reliable when true, but incomplete (unflagged ≠ legitimate).
2. Takeover labels sit on successful logins (to be verified below and in issue #3).
3. As the dataset authors state, per-user and global frequencies of the categorical features match the real
   service, even though the values themselves are synthetic.
4. Login order is preserved, although timestamps contain added randomness.
5. `Is Attack IP` is a *proxy* (an IP found in attacker lists), not a confirmed takeover.
6. A user ID identifies the same account across the whole period.
7. One row is one login attempt.
8. Behaviour of a 2020–2021 Norwegian SSO service is representative enough for a **methodological** study;
   the resulting model is not meant to be deployed as-is on another service.

**12. What is the data source?**

The *Login Data Set for Risk-Based Authentication* (Wiefling et al., 2022): login features synthesised from
the real behaviour of more than 3.3 million users of a large single sign-on service in Norway, collected
February 2020 – February 2021. Published on Zenodo (DOI
[10.5281/zenodo.6782156](https://doi.org/10.5281/zenodo.6782156), v1.0.0) under CC BY 4.0, with a mirror on
Kaggle (`dasgroup/rba-dataset`).

**13. How are the data updated?**

They are not: this is a static research release (v1.0.0, 2022). In a real deployment, login events would
stream continuously from the authentication service.

**14. How often are the data updated?**

Never, for this release. In production the stream is continuous, and how often a model must be retrained to
keep up with changing attack patterns (concept drift) is itself a question for later stages.

## 📚 Import libraries

In [1]:
import sys
from pathlib import Path

# Make the project's `src/` importable, wherever the notebook is opened from.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

In [2]:
import logging
import zipfile

import polars as pl

from data.datasets import load_dataset_config
from data.download import download_dataset, md5_of
from data.schema import FIELDS_BY_NAME, check_requirements

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(60)

polars.config.Config

## 💾 Load data

### Source

The dataset is described by its translation file, `conf/dataset/rba.yaml`. Loading it validates it: an
unknown key, a non-https URL, a malformed checksum or a mapping that misses the minimum requirements would
fail here.

In [3]:
config = load_dataset_config("rba")
source = config.source
pl.DataFrame(
    {
        "property": ["url", "doi", "version", "license", "archive", "published md5", "member"],
        "value": [
            str(source.url),
            source.doi,
            source.version,
            source.license,
            source.archive,
            source.md5,
            source.member,
        ],
    }
)

property,value
str,str
"""url""","""https://zenodo.org/api/records/6782156/files/rba-dataset.zip…"
"""doi""","""10.5281/zenodo.6782156"""
"""version""","""v1.0.0"""
"""license""","""CC-BY-4.0"""
"""archive""","""rba-dataset.zip"""
"""published md5""","""cc1b1078b3929650e6c08678caffcc57"""
"""member""","""rba-dataset.csv"""


### Download, verify, extract

`download_dataset` is idempotent. If a verified copy is already in `data/01_raw/` it is reused; otherwise the
archive is downloaded over https and its MD5 is checked against the published value. Before anything is
extracted, every entry of the archive is inspected — absolute paths, `..`, symbolic links, zip-bomb
compression ratios and oversized content are refused — and then only the expected CSV is extracted. Nothing
in `data/` is committed to git.

In [4]:
raw_dir = REPO_ROOT / "data" / "01_raw"
raw_file = download_dataset(config, raw_dir)
archive = raw_dir / source.archive

checksum = md5_of(archive)
print(f"archive : {archive.relative_to(REPO_ROOT)}  ({archive.stat().st_size / 1e9:.3f} GB)")
print(
    f"md5     : {checksum}  -> {'matches' if checksum == source.md5 else 'MISMATCH'} the published value"
)
print(f"raw file: {raw_file.relative_to(REPO_ROOT)}  ({raw_file.stat().st_size / 1e9:.2f} GB)")

INFO Already downloaded and verified: data/01_raw/rba-dataset.zip


INFO Already extracted: data/01_raw/rba-dataset.csv


archive : data/01_raw/rba-dataset.zip  (1.094 GB)
md5     : cc1b1078b3929650e6c08678caffcc57  -> matches the published value
raw file: data/01_raw/rba-dataset.csv  (9.05 GB)


In [5]:
with zipfile.ZipFile(archive) as zf:
    entries = pl.DataFrame(
        {
            "entry": [i.filename for i in zf.infolist()],
            "compressed_MB": [round(i.compress_size / 1e6, 1) for i in zf.infolist()],
            "uncompressed_MB": [round(i.file_size / 1e6, 1) for i in zf.infolist()],
        }
    )
entries

entry,compressed_MB,uncompressed_MB
str,f64,f64
"""LICENSE""",0.0,0.0
"""README.md""",0.0,0.0
"""RESULTS.md""",0.0,0.0
"""images/""",0.0,0.0
"""images/login-overview.png""",0.2,0.2
"""images/rq1-general.png""",0.5,0.5
"""images/rq1-login-frequency.png""",0.6,0.6
"""images/rq2-attack-data.png""",0.4,0.4
"""images/rq6-rtt-0_999.png""",0.4,0.4


## 🔎 First structural look

Only structure here: column names, a few raw rows, overall counts. Every column is read as text on purpose —
choosing proper types is the job of issue #2.

### Columns versus the translation file

In [6]:
raw = pl.scan_csv(raw_file, infer_schema=False)
header = raw.collect_schema().names()
rename = config.rename_map()

pl.DataFrame(
    {
        "column in file": header,
        "standard name": [rename.get(column, "— not mapped —") for column in header],
        "role": [
            FIELDS_BY_NAME[rename[column]].role.value if column in rename else "—"
            for column in header
        ],
    }
)

column in file,standard name,role
str,str,str
"""index""","""— not mapped —""","""—"""
"""Login Timestamp""","""timestamp""","""required"""
"""User ID""","""user_id""","""required"""
"""Round-Trip Time [ms]""","""rtt_ms""","""optional"""
"""IP Address""","""ip""","""context"""
"""Country""","""country""","""optional"""
"""Region""","""region""","""optional"""
"""City""","""city""","""optional"""
"""ASN""","""asn""","""context"""


In [7]:
missing_in_file = sorted(set(rename) - set(header))
not_mapped = sorted(set(header) - set(rename))
print("mapped columns missing from the file:", missing_in_file or "none")
print("file columns not mapped            :", not_mapped or "none")
assert not missing_in_file, "the translation file refers to columns the dataset does not have"

mapped columns missing from the file: none
file columns not mapped            : ['index']


### Minimum data requirements

The project needs `user_id`, `timestamp` and `login_success`, plus at least one context field
(`ip`, `asn`, `device_type`, `user_agent`) — see [`docs/datasets/README.md`](../../docs/datasets/README.md).

In [8]:
check_requirements(config.columns)
print(
    "Requirements met. Context fields available:",
    sorted(f for f in config.columns if FIELDS_BY_NAME[f].role.value == "context"),
)

Requirements met. Context fields available: ['asn', 'device_type', 'ip', 'user_agent']


### A few raw rows

In [9]:
raw.head(5).collect()

index,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,OS Name and Version,Device Type,Login Successful,Is Attack IP,Is Account Takeover
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""0""","""2020-02-03 12:43:30.772""","""-4324475583306591935""",null,"""10.0.65.171""","""NO""","""-""","""-""","""29695""","""Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like Mac OS X) Geck…","""Firefox 20.0.0.1618""","""iOS 13.4""","""mobile""","""False""","""False""","""False"""
"""1""","""2020-02-03 12:43:43.549""","""-4324475583306591935""",null,"""194.87.207.6""","""AU""","""-""","""-""","""60117""","""Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus Build/JRN84D)…","""Chrome Mobile 46.0.2490""","""Android 4.1""","""mobile""","""False""","""False""","""False"""
"""2""","""2020-02-03 12:43:55.873""","""-3284137479262433373""",null,"""81.167.144.58""","""NO""","""Vestland""","""Urangsvag""","""29695""","""Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) AppleWebKit/53…","""Android 2.3.3.2672""","""iOS 7.1""","""mobile""","""True""","""False""","""False"""
"""3""","""2020-02-03 12:43:56.180""","""-4324475583306591935""",null,"""170.39.78.152""","""US""","""-""","""-""","""393398""","""Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus Build/JRN84D)…","""Chrome Mobile WebView 85.0.4183""","""Android 4.1""","""mobile""","""False""","""False""","""False"""
"""4""","""2020-02-03 12:43:59.396""","""-4618854071942621186""",null,"""10.0.0.47""","""US""","""Virginia""","""Ashburn""","""398986""","""Mozilla/5.0 (Linux; U; Android 2.2) Build/NMA26.42-169; wv …","""Chrome Mobile WebView 85.0.4183""","""Android 2.2""","""mobile""","""False""","""True""","""False"""


### Overall counts

One streaming pass over the whole file. These numbers are checked against what the dataset authors
published, which is a second integrity check on top of the checksum.

In [10]:
# standard name -> column name in this file
column = {standard: in_file for in_file, standard in rename.items()}


def is_true(standard_name: str) -> pl.Expr:
    """Raw boolean columns are text here; treat 'true' (any case) as True."""
    return pl.col(column[standard_name]).str.to_lowercase() == "true"


success, ato, attack_ip = is_true("login_success"), is_true("label_ato"), is_true("label_attack_ip")

counts = raw.select(
    pl.len().alias("login_attempts"),
    pl.col(column["user_id"]).n_unique().alias("users"),
    success.sum().alias("successful"),
    (~success).sum().alias("failed"),
    ato.sum().alias("ato_total"),
    (ato & success).sum().alias("ato_on_successful"),
    (ato & ~success).sum().alias("ato_on_failed"),
    attack_ip.sum().alias("attack_ip_total"),
    pl.col(column["timestamp"]).min().alias("first_timestamp"),
    pl.col(column["timestamp"]).max().alias("last_timestamp"),
).collect(engine="streaming")

counts.transpose(include_header=True, header_name="measure", column_names=["value"])

measure,value
str,str
"""login_attempts""","""31269264"""
"""users""","""4304857"""
"""successful""","""12541442"""
"""failed""","""18727822"""
"""ato_total""","""141"""
"""ato_on_successful""","""140"""
"""ato_on_failed""","""1"""
"""attack_ip_total""","""3096977"""
"""first_timestamp""","""2020-02-03 12:43:30.772"""


In [11]:
c = counts.row(0, named=True)
print(f"takeover share among successful logins: {c['ato_on_successful'] / c['successful']:.6%}")
print(f"attack-IP share among all attempts    : {c['attack_ip_total'] / c['login_attempts']:.2%}")

takeover share among successful logins: 0.001116%
attack-IP share among all attempts    : 9.90%


### Comparison with the companion paper

Two counts differ from the paper (3.3 M users, 87 takeovers). Two candidate explanations are checked directly: *users* — do the extra IDs only appear in failed attempts? *takeovers* — is one incident spread over several rows?

In [12]:
users = raw.select(
    pl.col(column["user_id"]).n_unique().alias("all_user_ids"),
    pl.col(column["user_id"]).filter(success).n_unique().alias("users_with_a_success"),
    pl.col(column["user_id"]).filter(~success).n_unique().alias("users_with_a_failure"),
).collect(engine="streaming")

ato_rows = raw.filter(ato).collect(engine="streaming")
print(f"takeover rows: {ato_rows.height}, distinct users: {ato_rows[column['user_id']].n_unique()}")
print("users with N takeover rows:")
print(ato_rows.group_by(column["user_id"]).len().get_column("len").value_counts().sort("len"))
print(f"takeover rows that also carry an attack IP: {ato_rows.filter(attack_ip).height}")
users

takeover rows: 141, distinct users: 138
users with N takeover rows:
shape: (2, 2)
┌─────┬───────┐
│ len ┆ count │
│ --- ┆ ---   │
│ u32 ┆ u32   │
╞═════╪═══════╡
│ 1   ┆ 135   │
│ 2   ┆ 3     │
└─────┴───────┘
takeover rows that also carry an attack IP: 77


all_user_ids,users_with_a_success,users_with_a_failure
u32,u32,u32
4304857,3304502,2400077


## 📊 Analysis of Results and Conclusions

- **The download is verified and reproducible.** The archive matches the MD5 published on Zenodo
  (`cc1b1078…`), passed every archive-safety check, and its single data file expands to 9.05 GB. Re-running
  this notebook — or `make data` — reuses the verified copy.
- **The mapping is complete.** Every column named in the translation file exists in the CSV; the only
  unmapped column is `index`, a row number. The dataset meets the minimum data requirements with all four
  context fields available.
- **Volumes match the companion paper:** 31,269,264 login attempts (31.3 M), 12,541,442 successful (12.5 M)
  and 18,727,822 failed (18.8 M), from 2020-02-03 to 2021-02-28.
- **Users — explained.** There are 4,304,857 distinct user IDs, but 3,304,502 of them have at least one
  successful login, which is the paper's 3.3 M. The extra ~1 M IDs appear only in failed attempts,
  consistent with attackers trying accounts that do not exist (a hypothesis to examine in issue #3).
- **Takeovers — not explained.** The file has **141 takeover rows: 140 on successful logins and 1 on a
  failed login**, belonging to 138 users (only 3 users have two rows), while the paper reports *"87
  successful logins were detected as account takeover"* (§4.1). Counting incidents instead of rows does not
  close the gap, and neither the paper nor the dataset's README explains how the label was carried into the
  synthesised release. **The published file is used as the source of truth**, and the discrepancy is recorded
  as a limitation in the dataset card.
- **Assumption 2 holds:** 140 of 141 takeover labels sit on successful logins. The single takeover on a
  failed login falls outside the modelling population (successful logins) and is kept for EDA.
- **Extreme imbalance, as expected:** 0.0011% of successful logins are confirmed takeovers. A model that
  always answers "legitimate" would be right 99.999% of the time and catch nothing, which confirms the framing
  (question 4) and the metric choices (question 5).
- `Is Attack IP` marks 3,096,977 attempts (9.90%) — four orders of magnitude more than takeovers —
  consistent with a proxy for *attack attempts*, not confirmed takeovers. 77 of the 141 takeover rows also
  carry an attack IP.
- First data-quality notes for issue #2: `-` is used as a null marker (e.g. `Region`, `City`), round-trip
  time is often empty, and the very first rows already include a placeholder IP in `10.0.0.0/8`.

## 💡 Proposals and Ideas

- **Issue #2 (initial exploration):** load the CSV into the standard schema with proper types (timestamps,
  booleans, categorical identifiers — `asn` is an identifier, not a quantity), unify null representations,
  add the two artefact flags declared in the translation file (`asn_is_placeholder`, `ip_is_placeholder`),
  and store the result as parquet in `data/02_intermediate/`.
- **Issue #3 (EDA):** study the confirmed takeovers in depth — when they happen, which users, what preceded
  them (failed attempts, new devices or networks), and their overlap with attack IPs — and use their spread
  over time to decide the final test split by the rule fixed in the development plan.
- Fix a concrete operating target (question 7) once the Freeman baseline exists (issue #5).

## 📖 References

- Wiefling, S., Jørgensen, P. R., Thunem, S., & Lo Iacono, L. (2022). Pump Up Password Security! Evaluating
  and Enhancing Risk-Based Authentication on a Real-World Large-Scale Online Service. *ACM Transactions on
  Privacy and Security*, 26(1). https://doi.org/10.1145/3546069
- Wiefling, S. et al. (2022). *Login Data Set for Risk-Based Authentication* (v1.0.0) [Data set]. Zenodo.
  https://doi.org/10.5281/zenodo.6782156
- Freeman, D., Jain, S., Dürmuth, M., Biggio, B., & Giacinto, G. (2016). Who Are You? A Statistical Approach
  to Measuring User Authenticity. *NDSS 2016*. https://doi.org/10.14722/ndss.2016.23240
- Wiefling, S. (2022). *rba-algorithm* [Source code]. https://github.com/das-group/rba-algorithm
- Ren (2026); Sasikumar & Nagarajan (2026) — as cited in the anteproyecto.
- Course methodology: J. R. Zapata, *Ciencia de Datos en Producción* —
  https://joserzapata.github.io/post/ciencia-datos-proyecto-python/1-data/